# 5.3 TensorRT-LLM Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.3_tensorrt_llm/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.3_tensorrt_llm/lab.ipynb)

Hands-on exploration of TRT-LLM's compilation workflow, build command generation,
engine performance comparison, and a decision function for TRT-LLM vs vLLM selection.

In [ ]:
# Install dependencies via subprocess (Colab/Molab compatible)
import subprocess
import sys

# Install numpy and matplotlib for visualization
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])

In [ ]:
# === Core imports for TRT-LLM build and analysis experiments ===
# Core imports for the lab
import numpy as np  # numerical computation
import matplotlib.pyplot as plt  # plotting engine comparisons
from dataclasses import dataclass  # structured config objects
from typing import Optional, Dict, List  # type annotations

## Exercise 1: TRT-LLM Build Command Generator

Generate the correct `trtllm-build` CLI command for a given model configuration.
This mirrors the real deployment workflow: configure -> convert -> build -> serve.

In [ ]:
# === PARAMETERS: TRT-LLM build configuration ===
# Change these values and re-run to generate different build commands
# === PARAMETERS (change these and re-run) ===
MODEL_NAME = "meta-llama/Llama-3.1-70B"  # target model
TP_SIZE = 4  # tensor parallel degree (GPUs for layer splitting)
QUANTIZATION = "fp8"  # one of: None, "int8", "int4_awq", "fp8"
MAX_BATCH = 128  # max concurrent sequences
MAX_INPUT_LEN = 4096  # max prompt tokens
MAX_OUTPUT_LEN = 2048  # max generation tokens

In [ ]:
def generate_build_command(
    model: str,
    tp: int,
    quant: Optional[str],
    batch: int,
    input_len: int,
    output_len: int
) -> str:
    """Generate the trtllm-build CLI command for a given config."""
    # Start with required arguments
    parts = [
        "trtllm-build",
        "    --checkpoint_dir ./checkpoint/",  # converted weights location
        "    --output_dir ./engine/",  # where compiled engine lands
        f"    --max_batch_size {batch}",  # upper bound on concurrent seqs
        f"    --max_input_len {input_len}",  # max prompt length
        f"    --max_seq_len {input_len + output_len}",  # total context window
    ]
    # Add quantization-specific plugin flags
    # Conditional check
    if quant == "fp8":
        # Accumulate result
        parts.append("    --gemm_plugin fp8")  # use FP8 GEMM kernels
        # Accumulate result
        parts.append("    --gpt_attention_plugin fp8")  # FP8 attention
    # Alternative condition branch
    elif quant == "int4_awq":
        # Accumulate result
        parts.append("    --gemm_plugin auto")  # auto-select GEMM
    # Enable paged KV cache (always recommended)
    parts.append("    --use_paged_context_fmha enable")  # paged attention
    # Accumulate result
    parts.append("    --paged_kv_cache enable")  # paged KV storage
    # Accumulate result
    parts.append("    --tokens_per_block 64")  # KV page size in tokens
    # Parallel build workers speed up compilation
    parts.append(f"    --workers {min(tp, 4)}")  # parallel build threads
    # Return the computed result
    return " \\\n".join(parts)


# Generate and display the build command
cmd = generate_build_command(MODEL_NAME, TP_SIZE, QUANTIZATION, MAX_BATCH, MAX_INPUT_LEN, MAX_OUTPUT_LEN)
# Display results to user
print("Generated trtllm-build command:")
# Display results to user
print("=" * 50)
# Display results to user
print(cmd)

## Exercise 2: Build Time and Engine Size Estimation

Estimate compilation time and final engine size based on model parameters
and quantization choice. Useful for capacity planning CI/CD pipelines.

In [ ]:
def estimate_build_time_and_size(
    model_size_b: float,
    quant: Optional[str],
    tp: int
) -> Dict[str, float]:
    """Estimate build time (minutes) and engine size (GB)."""
    # Base build time per billion params (minutes)
    # FP8 needs calibration pass; INT4 AWQ needs more optimization
    time_per_b = {None: 1.1, "int8": 1.5, "int4_awq": 2.0, "fp8": 1.3}
    # Bytes per parameter after quantization
    bytes_per_param = {None: 2.0, "int8": 1.0, "int4_awq": 0.5, "fp8": 1.0}
    
    # Build time scales with model size, slightly reduced by parallelism
    build_min = time_per_b.get(quant, 1.1) * model_size_b / (tp ** 0.3)
    # Engine size = quantized weights + 10% overhead for metadata/kernels
    engine_gb = model_size_b * bytes_per_param.get(quant, 2.0) * 1.1
    # Per-GPU shard size
    shard_gb = engine_gb / tp
    
    # Return the computed result
    return {
        "build_time_min": round(build_min, 1),
        "total_engine_gb": round(engine_gb, 1),
        "per_gpu_gb": round(shard_gb, 1),
        "num_shards": tp
    }


# Compare across model sizes and quantizations
models = [("8B", 8), ("70B", 70), ("405B", 405)]  # name, size in billions
quants = [None, "fp8", "int4_awq"]  # quantization options to compare

# Display results to user
print(f"{'Model':<8} {'Quant':<10} {'TP':<4} {'Build(min)':<12} {'Engine(GB)':<12} {'Per-GPU(GB)'}")
# Display results to user
print("-" * 60)
# Iterate over each item
for name, size in models:
    # Choose TP based on model size (heuristic)
    tp = 1 if size <= 8 else (4 if size <= 70 else 8)
    # Iterate over each item
    for q in quants:
        # Estimate build metrics for this configuration
        est = estimate_build_time_and_size(size, q, tp)
        q_label = q if q else "fp16"  # display label
        # Display results to user
        print(f"{name:<8} {q_label:<10} {tp:<4} {est['build_time_min']:<12} {est['total_engine_gb']:<12} {est['per_gpu_gb']}")

## Exercise 3: Engine Throughput Comparison Chart

Visualize TRT-LLM vs vLLM vs SGLang throughput across batch sizes.
TRT-LLM's advantage grows with batch size due to compiled memory management.

In [ ]:
def throughput_model(
    batch_sizes: np.ndarray,
    peak_tps: float,
    half_batch: float
) -> np.ndarray:
    """Model throughput as saturating function of batch size.
    Uses logistic curve: throughput = peak * batch / (batch + half_batch)
    """
    # Saturating throughput curve (GPU utilization increases then plateaus)
    # Return the computed result
    return peak_tps * batch_sizes / (batch_sizes + half_batch)


# Batch sizes to evaluate (1 to 256)
batches = np.arange(1, 257)  # range of concurrent sequences

# Throughput curves for each engine (Llama 70B, FP8, 4xH100)
# Peak throughput and half-saturation batch from published benchmarks
trt_tps = throughput_model(batches, peak_tps=14500, half_batch=16)  # TRT-LLM: highest peak
vllm_tps = throughput_model(batches, peak_tps=11500, half_batch=20)  # vLLM: good but lower
sglang_tps = throughput_model(batches, peak_tps=12500, half_batch=18)  # SGLang: middle ground

# Create the comparison chart
# Configure plot element
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
# Plot data series
ax.plot(batches, trt_tps, 'b-', linewidth=2, label='TRT-LLM 1.0')  # blue for TRT-LLM
ax.plot(batches, vllm_tps, 'r--', linewidth=2, label='vLLM 0.6')  # red dashed for vLLM
ax.plot(batches, sglang_tps, 'g-.', linewidth=2, label='SGLang 0.3')  # green dash-dot

# Mark the key benchmark point (batch=128)
ax.axvline(x=128, color='gray', linestyle=':', alpha=0.5)  # reference line
ax.annotate('batch=128\n(benchmark point)', xy=(128, 1000), fontsize=9, color='gray')  # label

# Formatting
ax.set_xlabel('Batch Size', fontsize=12)  # x-axis label
ax.set_ylabel('Throughput (tokens/sec)', fontsize=12)  # y-axis label
# Configure plot element
ax.set_title('Engine Throughput vs Batch Size\n(Llama 70B, FP8, 4xH100 SXM)', fontsize=13)
ax.legend(fontsize=11)  # show engine legend
ax.grid(True, alpha=0.3)  # subtle grid for readability
ax.set_xlim(0, 260)  # x range
ax.set_ylim(0, 15000)  # y range
plt.tight_layout()  # prevent label clipping
# Configure plot element
plt.show()

## Exercise 4: TRT-LLM vs vLLM Decision Function

A scoring-based decision function that recommends TRT-LLM or vLLM
based on workload requirements. Encodes the tradeoffs from the module.

In [ ]:
@dataclass
class WorkloadSpec:
    """Describes deployment requirements for engine selection."""
    model_size_b: float  # model size in billions of params
    target_throughput: float  # desired tokens/sec
    max_itl_ms: float  # max acceptable inter-token latency
    num_gpus: int  # available GPU count
    model_changes_weekly: bool  # do you swap models often?
    nvidia_only: bool  # is the fleet 100% NVIDIA?
    need_fp8: bool  # is FP8 quantization required?


def decide_engine(spec: WorkloadSpec) -> Dict:
    """Score-based engine recommendation."""
    # Initialize scores for each engine
    trt_score = 0  # TRT-LLM advantage accumulator
    vllm_score = 0  # vLLM advantage accumulator
    trt_reasons = []  # explanations for TRT-LLM points
    vllm_reasons = []  # explanations for vLLM points

    # Tight latency budget strongly favors compiled kernels
    # Conditional check
    if spec.max_itl_ms < 15:
        trt_score += 3
        trt_reasons.append("Tight ITL (<15ms) needs kernel fusion")

    # Frequent model changes penalize the build step
    # Conditional check
    if spec.model_changes_weekly:
        vllm_score += 3
        vllm_reasons.append("Weekly model swaps: build step too costly")

    # NVIDIA-only fleet unlocks hardware-specific optimizations
    # Conditional check
    if spec.nvidia_only:
        trt_score += 2
        trt_reasons.append("NVIDIA-only: full hardware exploitation")
    else:
        vllm_score += 2
        vllm_reasons.append("Multi-vendor fleet: need portable engine")

    # FP8 requirement: TRT-LLM's implementation is most mature
    # Conditional check
    if spec.need_fp8:
        trt_score += 2
        trt_reasons.append("FP8 quantization: TRT-LLM most mature")

    # High throughput target favors compiled optimizations
    throughput_per_gpu = spec.target_throughput / spec.num_gpus
    # Conditional check
    if throughput_per_gpu > 2000:
        trt_score += 2
        trt_reasons.append(f"High per-GPU throughput ({throughput_per_gpu:.0f} tok/s)")

    # Determine winner (vLLM wins ties: simpler to operate)
    # Conditional check
    if trt_score > vllm_score:
        winner = "TRT-LLM"
        reasons = trt_reasons
    else:
        winner = "vLLM"
        reasons = vllm_reasons
        # Conditional check
        if trt_score == vllm_score:
            reasons.append("Tie-breaker: vLLM simpler to operate")

    # Return the computed result
    return {"recommendation": winner, "trt_score": trt_score, "vllm_score": vllm_score, "reasons": reasons}


# Scenario A: Latency-critical production chat
spec_a = WorkloadSpec(
    model_size_b=70, target_throughput=10000, max_itl_ms=10,
    num_gpus=4, model_changes_weekly=False, nvidia_only=True, need_fp8=True
)
result_a = decide_engine(spec_a)
# Display results to user
print("Scenario A: Production chat (70B, 4xH100, tight latency)")
# Display results to user
print(f"  -> {result_a['recommendation']} (score {result_a['trt_score']} vs {result_a['vllm_score']})")
# Iterate over each item
for r in result_a['reasons']:
    # Display results to user
    print(f"     {r}")

# Display results to user
print()

# Scenario B: Research team with frequent model swaps
spec_b = WorkloadSpec(
    model_size_b=8, target_throughput=2000, max_itl_ms=50,
    num_gpus=1, model_changes_weekly=True, nvidia_only=True, need_fp8=False
)
result_b = decide_engine(spec_b)
# Display results to user
print("Scenario B: Research iteration (8B, 1 GPU, weekly model swaps)")
# Display results to user
print(f"  -> {result_b['recommendation']} (score {result_b['trt_score']} vs {result_b['vllm_score']})")
# Iterate over each item
for r in result_b['reasons']:
    # Display results to user
    print(f"     {r}")

## Key Takeaways

- TRT-LLM wins on raw throughput and latency via AOT compilation
- Build time is the primary operational cost (8 min to 3 hours)
- vLLM wins on flexibility, iteration speed, and operational simplicity
- The decision hinges on: model stability, latency budget, hardware homogeneity